# Lab 5 Report: 
## Create Arthur Conan Doyle AI with RNN

### Name:

In [ ]:
%matplotlib inline

import numpy as np
import matplotlib.pyplot as plt
import torch
from torch.distributions import Categorical
import math;

In [ ]:
from IPython.display import Image # For displaying images in colab jupyter cell

In [ ]:
Image('lab5_exercise.png', width = 1000)

## Prepare Data

In [ ]:
# You will train on the first N characters of the Sherlock Holmes book
# Pick the size of your training data, i.e. N



# Load the Sherlock Holmes data up to data_size_to_train
data_size_to_train = 20000;
use_your_words = True;
data = list(open("sherlock.txt").read())[:data_size_to_train]# YOUR CODE HERE
#data_size_to_train = len(data)# YOUR CODE HERE
# Find the set of unique characters within the training data
characters = sorted(list(set(data)))# YOUR CODE HERE

chars = str(open("sherlock.txt").read());
words  = set(chars.split());
data_size, vocab_size = len(data), len(characters)
# total number of characters in the training data and number of unique characters
data_size, vocab_size = len(data), len(characters)

print("Data has {} characters, {} unique".format(data_size, vocab_size))
print(words);

In [ ]:
# Use Python Dictionary to map the characters to numbers and vice versa

# YOUR CODE HERE

chars_to_nums = {ch: i for i, ch in enumerate(characters)};
nums_to_chars = {i:ch for i, ch in enumerate(characters)};
print(chars_to_nums);

In [ ]:
# Use the character_to_num dictionary to map each character in the training dataset to a number

# YOUR CODE HERE
data= list(data);
for i, ch in enumerate(data):
    data[i] = chars_to_nums[ch];

print(data[:32]);

def ctrl_alt_del(output, st:set[str]):
    to_return =0;
    output= torch.nn.functional.softmax(torch.squeeze(output), dim = 0);
    character_distribution = torch.distributions.Categorical(output);
    character_num = character_distribution.sample();
    value = "";
    for i in range(character_num.shape[0]):
        value += nums_to_chars[int(character_num[i])];
    for i in value.split():
        if not i in st:
            to_return += 1;

    return to_return;

def ctrl_alt_del_str(value, st:set[str]):
    to_return =0;
    for i in value.split():
        if not i in st:
            to_return += 1;

    return to_return;


## Define Model

In [ ]:
class CharRNN(torch.nn.Module):
    
    def __init__(self, num_embeddings, embedding_dim, input_size, hidden_size, num_layers, output_size):
        
        super(CharRNN, self).__init__()
        # YOUR CODE HERE
        self.embedding = torch.nn.Embedding(num_embeddings, embedding_dim); 
        self.rnn = torch.nn.RNN(input_size = input_size, hidden_size =hidden_size, num_layers = num_layers, nonlinearity = "relu");
        self.decoder = torch.nn.Linear(hidden_size, output_size);

    def forward(self, input_seq, hidden_state):
        # YOUR CODE HERE
        embedding = self.embedding(input_seq);
        output, hidden_state = self.rnn(embedding, hidden_state);
        output = self.decoder(output);
        return output, hidden_state.detach();


## Define Hyperparameters

In [ ]:
# Fix random seed
torch.manual_seed(25)

# Define RNN network
rnn = CharRNN(num_embeddings= vocab_size, embedding_dim=100, input_size= 100, hidden_size= 512, num_layers= 3, output_size= vocab_size)

# Define learning rate and epochs
learning_rate = 0.001# YOUR CODE HERE 
epochs = 500# YOUR CODE HERE

# Size of the input sequence to be used during training and validation
training_sequence_len = 200# YOUR CODE HERE
validation_sequence_len = 500# YOUR CODE HERE    

word_div = 4.;
# Define loss function and optimizer
loss_fn = torch.nn.CrossEntropyLoss()# YOUR CODE HERE
optimizer = torch.optim.Adam(rnn.parameters(), lr = learning_rate)# YOUR CODE HERE

# add .cuda() for GPU acceleration
rnn

## Identify Tracked Values

In [ ]:
# Tracking training loss per each input/target sequence fwd/bwd pass
# YOUR CODE HERE
training_loss_list = []
loss_2_list = []

## Train Model

In [ ]:
# Convert training data into torch tensor and make it into vertical orientation (N, 1)
# Attach .cuda() if using GPU
data = torch.unsqueeze(torch.tensor(data), dim = 1)# YOUR CODE HERE

# Training Loop ----------------------------------------------------------------------------------------------------------

for epoch in range(epochs):
    character_loc = np.random.randint(data_size_to_train);
    iteration = 0;
    hidden_state = None;
    while character_loc+training_sequence_len+1 <data_size:
        input_seq = data[character_loc: character_loc+training_sequence_len]
        target_seq = data[character_loc+1:character_loc+training_sequence_len+1]
        output, hidden_state = rnn(input_seq, hidden_state);
        loss = loss_fn(torch.squeeze(output), torch.squeeze(target_seq));
        loss2 = math.sqrt(ctrl_alt_del( output,words))/word_div;
        loss_2_list.append(loss2);
        if use_your_words:
            loss += loss2;
        training_loss_list.append(loss.item());
        optimizer.zero_grad();
        loss.backward();
        optimizer.step();
        character_loc += training_sequence_len;
        iteration += 1
    print("Averaged Training Loss for Epoch", epoch, ": ", np.mean(training_loss_list), "average number words:",np.mean(loss_2_list));
    # YOUR CODE HERE
    
    # Sample and generate a text sequence after every epoch --------------------------------------------------------------
    character_loc = 0;
    hidden_state = None;
    rand_indx = np.random.randint(data_size-1);
    input_seq = data[rand_indx:rand_indx+1];
    # YOUR CODE HERE
    print("----------------------------------------")
    with torch.no_grad():
        while character_loc <validation_sequence_len:
            output, hidden_state = rnn(input_seq, hidden_state);
            output = torch.nn.functional.softmax(torch.squeeze(output), dim = 0);
            character_distribution = torch.distributions.Categorical(output);
            character_num = character_distribution.sample();
            print(nums_to_chars[int(character_num.item())], end = '')
            input_seq[0][0] = character_num.item();
            character_loc+=1;
    # YOUR CODE HERE

    print("\n----------------------------------------")

## Visualize & Evaluate Model

In [ ]:
# Print a validation text sequence that most closely resembles Sherlock Holmes style

validation_loss_list = []
validation_strings = []
min_idx = -1;
min_loss = 0;
validation_try_count = 100;
with torch.no_grad():
    for i in range(validation_try_count):
        hidden_state = None;
        base_idx = np.random.randint(data_size_to_train-validation_sequence_len);
        input_seq= data[base_idx: base_idx+1];
        valid_seq = data[base_idx:base_idx+validation_sequence_len];
        output_seq = [];
        st = "";
        for j in range(validation_sequence_len):
                #print("<",st,">");
                #print(list(st)[j:validation_sequence_len+j], j)
                output, hidden_state = rnn(input_seq, hidden_state);
                output = torch.nn.functional.softmax(torch.squeeze(output), dim = 0);
                character_distribution = torch.distributions.Categorical(output);
                character_num = character_distribution.sample();
                input_seq[0][0] = character_num.item();
                output_seq.append(character_num.item());
                st += nums_to_chars[int(character_num.item())]
        loss0 =loss_fn(torch.squeeze(torch.tensor(output_seq)).float(),torch.squeeze(valid_seq).float());
        loss  = ctrl_alt_del_str(st, words))/word_div;
        if use_your_words:
            loss0 += loss;
        #print("epoch:", i, "value:",st, "loss:",loss );
        validation_strings.append(st);
        validation_loss_list.append(loss0.item());
        if min_idx == -1 or min_loss>loss0.item():
            min_idx = i;
            min_loss = loss0.item();
print("best:",validation_strings[min_idx], "error:",validation_loss_list[min_idx])


In [ ]:
# Import seaborn for prettier plot
import seaborn as sns

sns.set(style = 'whitegrid', font_scale = 2.5)

In [ ]:
# Plot the training loss and rolling mean training loss with respect to iterations
# Feel free to change the window size
plt.figure(figsize = (15, 9))

plt.plot(training_loss_list, linewidth = 3, label = 'Training Loss')
plt.plot(np.convolve(training_loss_list, np.ones(100), 'valid') / 100, 
         linewidth = 3, label = 'Rolling Averaged Training Loss')
plt.ylabel("training loss")
plt.xlabel("Iterations")
plt.legend()
sns.despine()